# 03 — Experimentación del módulo Bayesiano

Estimación de CPTs sobre el subset `label == 'coffee'` de `crop_recommendation.csv`, ejecución de Variable Elimination desde cero (`src/bayesian/inference.py`) y validación contra `pgmpy`.

**DAG:** `T, H, P → R → Y`.

**Estados:** todas las variables discretizadas en `{0: bajo/baja, 1: medio/media, 2: alto/alta}`.

**Etiquetado sintético:** `crop_recommendation.csv` no incluye etiqueta de roya ni de rendimiento. `R` se deriva determinísticamente de `(T, H, P)` por una regla agronómica (Avelino et al. sobre *Hemileia vastatrix*); `Y` se muestrea condicionado a `R` con distribuciones fijas y `seed=42`. Las reglas están en `src/bayesian/cpt_estimator.py`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.bayesian.cpt_estimator import build_labeled_dataset, estimate_cpt
from src.bayesian.network import build_coffee_network
from src.bayesian.inference import variable_elimination

raw = pd.read_csv('../data/raw/crop_recommendation.csv')
coffee = raw[raw['label'] == 'coffee'].copy()
len(coffee), coffee[['temperature', 'humidity', 'rainfall']].describe().round(2)

(100,
        temperature  humidity  rainfall
 count       100.00    100.00    100.00
 mean         25.54     58.87    158.07
 std           1.50      5.84     25.70
 min          23.06     50.05    115.16
 25%          24.22     53.81    136.01
 50%          25.66     57.65    157.77
 75%          26.74     63.58    181.47
 max          27.92     69.95    199.47)

## 1. Etiquetado sintético del subset

In [2]:
labeled = build_labeled_dataset(coffee, seed=42)
labeled.head(10)

,T,H,P,R,Y
2100,2,0,0,0,2
2101,2,0,0,0,2
2102,2,0,0,0,2
2103,2,0,0,0,2
2104,1,1,0,1,0
2105,1,0,0,1,2
2106,1,0,0,1,1
2107,1,1,0,1,1
2108,2,0,0,0,1
2109,2,1,0,1,1


In [3]:
for col in ['T', 'H', 'P', 'R', 'Y']:
    dist = labeled[col].value_counts(normalize=True).sort_index().round(3)
    print(f'{col}: {dist.to_dict()}')

T: {1: 0.21, 2: 0.79}
H: {0: 0.58, 1: 0.42}
P: {0: 1.0}
R: {0: 0.48, 1: 0.52}
Y: {0: 0.11, 1: 0.45, 2: 0.44}


## 2. Estimación de CPTs con Laplace (α=1)

In [4]:
card = {'T': 3, 'H': 3, 'P': 3, 'R': 3, 'Y': 3}
T_cpt = estimate_cpt(labeled, 'T', [], card)
H_cpt = estimate_cpt(labeled, 'H', [], card)
P_cpt = estimate_cpt(labeled, 'P', [], card)
R_cpt = estimate_cpt(labeled, 'R', ['T', 'H', 'P'], card)
Y_cpt = estimate_cpt(labeled, 'Y', ['R'], card)

print('P(T):', T_cpt.round(3))
print('P(H):', H_cpt.round(3))
print('P(P):', P_cpt.round(3))
print('shape R|T,H,P:', R_cpt.shape, ' suma por padre =', R_cpt.sum(axis=0).round(2))
print('shape Y|R:', Y_cpt.shape, ' suma por padre =', Y_cpt.sum(axis=0).round(2))

P(T): [0.01  0.214 0.777]
P(H): [0.573 0.417 0.01 ]
P(P): [0.981 0.01  0.01 ]
shape R|T,H,P: (3, 3, 3, 3)  suma por padre = [[[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]]
shape Y|R: (3, 3)  suma por padre = [1. 1. 1.]


In [5]:
y_table = pd.DataFrame(Y_cpt, index=['Y=bajo', 'Y=medio', 'Y=alto'], columns=['R=bajo', 'R=medio', 'R=alto']).round(3)
y_table

,R=bajo,R=medio,R=alto
Y=bajo,0.039,0.200,0.333
Y=medio,0.235,0.636,0.333
Y=alto,0.725,0.164,0.333


## 3. Inferencia con la implementación propia

Variable Elimination paso a paso: reducción por evidencia, producto de factores con alineación por nombre de variable, marginalización por suma.

In [6]:
net = build_coffee_network(T_cpt, H_cpt, P_cpt, R_cpt, Y_cpt)

casos = [
    ('marginal_R',              ['R'], {}),
    ('marginal_Y',              ['Y'], {}),
    ('R | T=alta',              ['R'], {'T': 2}),
    ('R | H=alta, P=media',     ['R'], {'H': 2, 'P': 1}),
    ('Y | T=media, H=alta',     ['Y'], {'T': 1, 'H': 2}),
    ('R,Y | T=alta, H=baja',    ['R', 'Y'], {'T': 2, 'H': 0}),
]

rows = []
for label, q, ev in casos:
    res = variable_elimination(net, q, ev)
    rows.append({'caso': label, 'vars': res.variables, 'shape': res.array.shape, 'suma': float(res.array.sum().round(4))})
pd.DataFrame(rows)

,caso,vars,shape,suma
0,marginal_R,[R],"(3,)",1.0
1,marginal_Y,[Y],"(3,)",1.0
2,R | T=alta,[R],"(3,)",1.0
3,"R | H=alta, P=media",[R],"(3,)",1.0
4,"Y | T=media, H=alta",[Y],"(3,)",1.0
5,"R,Y | T=alta, H=baja","[Y, R]","(3, 3)",1.0


In [7]:
post_r = variable_elimination(net, ['R'], {'T': 2, 'H': 2})
post_y = variable_elimination(net, ['Y'], {'T': 2, 'H': 2})
print('Evidencia: T=alta, H=alta')
print(f'  P(R) = bajo:{post_r.array[0]:.3f}  medio:{post_r.array[1]:.3f}  alto:{post_r.array[2]:.3f}')
print(f'  P(Y) = bajo:{post_y.array[0]:.3f}  medio:{post_y.array[1]:.3f}  alto:{post_y.array[2]:.3f}')

Evidencia: T=alta, H=alta
  P(R) = bajo:0.333  medio:0.333  alto:0.333
  P(Y) = bajo:0.191  medio:0.402  alto:0.407


## 4. Validación contra `pgmpy`

Se construye la misma red con `pgmpy.models.DiscreteBayesianNetwork` cargando las mismas CPTs estimadas, se ejecuta `VariableElimination` y se compara término a término contra la implementación propia. Diferencia máxima esperada: < 1e-9 (mismas CPTs → misma factorización exacta).

In [8]:
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination as PgmpyVE

model = BayesianNetwork([("T", "R"), ("H", "R"), ("P", "R"), ("R", "Y")])

def to_tabular(name, arr, parents=None):
    if parents:
        values = arr.reshape(card[name], -1)
        return TabularCPD(
            variable=name, variable_card=card[name], values=values,
            evidence=parents, evidence_card=[card[p] for p in parents],
        )
    return TabularCPD(variable=name, variable_card=card[name], values=arr.reshape(-1, 1))

model.add_cpds(
    to_tabular("T", T_cpt),
    to_tabular("H", H_cpt),
    to_tabular("P", P_cpt),
    to_tabular("R", R_cpt, ["T", "H", "P"]),
    to_tabular("Y", Y_cpt, ["R"]),
)
model.check_model()

/home/tunchxz/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/tunchxz/.local/lib/python3.8/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.8.10). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/tunchxz/.local/lib/python3.8/site-packages/google/auth/__init__.py:52: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.8"), FutureWarning)
/home/tunchxz/.local/lib/python3.8/site-packages/google/oauth2/__init__.py:38: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with critical bug fixes

True

In [9]:
pgmpy_infer = PgmpyVE(model)

def compare(query, evidence):
    mine = variable_elimination(net, query, evidence)
    ref = pgmpy_infer.query(variables=list(query), evidence=evidence, show_progress=False)
    # Alinear ejes a mismo orden
    ref_arr = ref.values
    # pgmpy retorna en el orden de query; nuestra impl puede devolver otro orden
    perm = [mine.variables.index(v) for v in query]
    mine_arr = mine.array.transpose(perm) if mine.array.ndim > 1 else mine.array
    diff = np.max(np.abs(mine_arr - ref_arr))
    return mine_arr, ref_arr, diff

rows = []
for label, q, ev in casos:
    _, _, diff = compare(q, ev)
    rows.append({'caso': label, 'diff_max_abs': float(diff)})
pd.DataFrame(rows)

,caso,diff_max_abs
0,marginal_R,5.551115e-17
1,marginal_Y,5.551115e-17
2,R | T=alta,1.110223e-16
3,"R | H=alta, P=media",0.000000e+00
4,"Y | T=media, H=alta",2.775558e-17
5,"R,Y | T=alta, H=baja",2.220446e-16


## 5. Análisis de sensibilidad

Cómo varía `P(RiesgoRoya = alto)` al barrer la evidencia climática. La red propaga los efectos de `T, H, P` a `R` y de `R` a `Y`.

In [10]:
import itertools

names = ['baja', 'media', 'alta']
rows = []
for t, h, p in itertools.product([0, 1, 2], repeat=3):
    post = variable_elimination(net, ['R'], {'T': t, 'H': h, 'P': p}).array
    y_post = variable_elimination(net, ['Y'], {'T': t, 'H': h, 'P': p}).array
    rows.append({
        'T': names[t], 'H': names[h], 'P': names[p],
        'P(R=bajo)': round(float(post[0]), 3),
        'P(R=medio)': round(float(post[1]), 3),
        'P(R=alto)': round(float(post[2]), 3),
        'P(Y=alto)': round(float(y_post[2]), 3),
    })
sens = pd.DataFrame(rows).sort_values('P(R=alto)', ascending=False)
sens.head(15)

,T,H,P,P(R=bajo),P(R=medio),P(R=alto),P(Y=alto)
0,baja,baja,baja,0.333,0.333,0.333,0.407
1,baja,baja,media,0.333,0.333,0.333,0.407
25,alta,alta,media,0.333,0.333,0.333,0.407
24,alta,alta,baja,0.333,0.333,0.333,0.407
23,alta,media,alta,0.333,0.333,0.333,0.407
22,alta,media,media,0.333,0.333,0.333,0.407
20,alta,baja,alta,0.333,0.333,0.333,0.407
19,alta,baja,media,0.333,0.333,0.333,0.407
17,media,alta,alta,0.333,0.333,0.333,0.407
16,media,alta,media,0.333,0.333,0.333,0.407
